In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
%scala
import org.apache.spark.sql.functions._

// Parametros (los widgets se definen en la celda 3)
val catalogo      = dbutils.widgets.get("catalogo")
val esquemaSource = dbutils.widgets.get("esquema_source")
val esquemaGold   = "golden"   // esquema gold existente en catalog_au

// 1. Lectura de las tres fuentes bronze con limpieza basica
val dfModelos = spark.table(s"$catalogo.$esquemaSource.modelos_ia")
  .select(
    trim(col("id_modelo")).as("id_modelo"),
    trim(col("nombre_modelo")).as("nombre_modelo"),
    trim(col("tipo_modelo")).as("tipo_modelo"),
    trim(col("empresa")).as("empresa"),
    col("anio_lanzamiento"))
  .filter(col("id_modelo").isNotNull)
  .dropDuplicates("id_modelo")

val dfAplicaciones = spark.table(s"$catalogo.$esquemaSource.aplicaciones_ia")
  .select(
    trim(col("id_aplicacion")).as("id_aplicacion"),
    trim(col("nombre_aplicacion")).as("nombre_aplicacion"),
    trim(col("sector")).as("sector"),
    trim(col("id_modelo")).as("id_modelo"))
  .filter(col("id_aplicacion").isNotNull && col("id_modelo").isNotNull)
  .dropDuplicates("id_aplicacion")

val dfUsuarios = spark.table(s"$catalogo.$esquemaSource.usuarios_aplicaciones")
  .select(
    trim(col("id_usuario")).as("id_usuario"),
    trim(col("nombre_usuario")).as("nombre_usuario"),
    trim(col("pais")).as("pais"),
    trim(col("id_aplicacion")).as("id_aplicacion"),
    col("uso_mensual_horas"))
  // Filtros: claves validas y uso positivo
  .filter(col("id_usuario").isNotNull && col("id_aplicacion").isNotNull)
  .filter(col("uso_mensual_horas").isNotNull && col("uso_mensual_horas") > 0)
  .dropDuplicates("id_usuario")

// 2. Cruces: usuarios -> aplicaciones -> modelos
val dfGold = dfUsuarios
  .join(dfAplicaciones, Seq("id_aplicacion"), "inner")
  .join(dfModelos, Seq("id_modelo"), "inner")
  // 3. Columnas derivadas para el tablero
  .withColumn("categoria_uso",
    when(col("uso_mensual_horas") < 20, "Bajo")
      .when(col("uso_mensual_horas") < 50, "Medio")
      .otherwise("Alto"))
  .withColumn("antiguedad_modelo_anios", year(current_date()) - col("anio_lanzamiento"))
  .withColumn("fecha_carga", current_timestamp())
  .select(
    "id_usuario", "nombre_usuario", "pais", "uso_mensual_horas", "categoria_uso",
    "id_aplicacion", "nombre_aplicacion", "sector",
    "id_modelo", "nombre_modelo", "tipo_modelo", "empresa", "anio_lanzamiento", "antiguedad_modelo_anios",
    "fecha_carga")

println(s"Filas resultantes: ${dfGold.count()}")
display(dfGold)

In [0]:
%scala
// 4. Persistencia en el esquema gold para el tablero
val tablaGold = s"$catalogo.$esquemaGold.uso_aplicaciones_ia"

dfGold.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(tablaGold)

// 5. Validacion
val dfCheck = spark.table(tablaGold)
println(s"Tabla $tablaGold -> filas: ${dfCheck.count()}, usuarios unicos: ${dfCheck.select("id_usuario").distinct().count()}")
display(dfCheck.groupBy("sector", "tipo_modelo").agg(count("*").as("usuarios"), round(avg("uso_mensual_horas"), 2).as("uso_promedio_horas")).orderBy(desc("usuarios")))